# RetainIQ — Phase 7.1: Business Analysis Dataset & Intervention Input

## Objective

I bring together the segment and geographic retention outputs from Phases 5 and 6 and prepare the business-analysis layer for prioritization.

I will also create the editable intervention-scoring template required by the business-analysis roadmap. The score is a **subjective 1–5 business input**, so I will not invent it inside the analytical logic.

## What I will produce

- A validated segment strategy dataset
- Validated state and city market datasets
- A self-contained intervention-score template
- A clean combined exposure view for later prioritization

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. Setup-only cells are kept clean so the notebook does not become repetitive.

In [1]:
from getpass import getpass
from pathlib import Path
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq",
}

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"
CONFIG_DIR = ROOT / "config"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)


def run_query(query, params=None):
    connection = None
    cursor = None
    try:
        connection = get_connection()
        cursor = connection.cursor(dictionary=True)
        cursor.execute(query, params or ())
        rows = cursor.fetchall()
        return pd.DataFrame(rows)
    except Error as exc:
        raise RuntimeError(f"MySQL query failed: {exc}") from exc
    finally:
        if cursor is not None:
            cursor.close()
        if connection is not None and connection.is_connected():
            connection.close()


def print_result(message):
    print(f"Result: {message}")

In [4]:
required_tables = [
    "segment_profile_summary",
    "state_retention_profile",
    "city_retention_profile",
]

check_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'retainiq'
  AND table_name IN (%s, %s, %s)
ORDER BY table_name;
"""

existing = run_query(check_query, tuple(required_tables))
if not existing.empty:
    existing.columns = [col.lower() for col in existing.columns]

existing_tables = existing["table_name"].tolist() if not existing.empty else []
missing_tables = [t for t in required_tables if t not in existing_tables]

if missing_tables:
    raise RuntimeError(
        "Phase 7 requires these MySQL objects first: " + ", ".join(missing_tables) +
        ". Run the Phase 5 and Phase 6 publishing notebooks before starting Phase 7."
    )

print_result(f"All Phase 7 source tables are available: {', '.join(existing_tables)}.")

Result: All Phase 7 source tables are available: city_retention_profile, segment_profile_summary, state_retention_profile.


### Result & conclusion

I have confirmed that the Phase 5 and Phase 6 reporting tables required by Phase 7 are available in MySQL. This means I can build the business-analysis layer from the published project outputs rather than duplicating earlier transformations.

In [5]:
segment_query = """
SELECT
    segment_id,
    segment_name,
    retention_profile,
    customers,
    share_pct,
    avg_tenure_months,
    avg_monthly_charge,
    avg_cltv,
    avg_satisfaction,
    churned_customers,
    churn_rate_pct,
    revenue_at_risk
FROM segment_profile_summary
ORDER BY segment_id;
"""

state_query = """
SELECT
    state,
    customers,
    churned_customers,
    avg_cltv,
    total_revenue,
    avg_monthly_charge,
    avg_satisfaction,
    churn_rate_pct,
    revenue_at_risk,
    share_of_customers_pct
FROM state_retention_profile
ORDER BY state;
"""

city_query = """
SELECT
    state,
    city,
    customers,
    churned_customers,
    avg_cltv,
    total_revenue,
    avg_monthly_charge,
    avg_satisfaction,
    churn_rate_pct,
    revenue_at_risk
FROM city_retention_profile
ORDER BY state, city;
"""

segment_df = run_query(segment_query)
state_df = run_query(state_query)
city_df = run_query(city_query)

numeric_cols = [
    "share_pct", "avg_tenure_months", "avg_monthly_charge", "avg_cltv",
    "avg_satisfaction", "churn_rate_pct", "revenue_at_risk", "total_revenue",
    "share_of_customers_pct"
]
for df in [segment_df, state_df, city_df]:
    for column in numeric_cols:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

print_result(
    f"Loaded {len(segment_df):,} segment rows, {len(state_df):,} state rows, and {len(city_df):,} city rows."
)
display(segment_df.head())
display(state_df.head())
display(city_df.head())

Result: Loaded 3 segment rows, 1 state rows, and 1,106 city rows.


,segment_id,segment_name,retention_profile,customers,share_pct,avg_tenure_months,avg_monthly_charge,avg_cltv,avg_satisfaction,churned_customers,churn_rate_pct,revenue_at_risk
0,0,Segment 0 — Emerging Risk,Retention Priority,3226,45.80,15.74,69.78,4013.45,2.81,1516,46.99,1944581.19
1,1,Segment 1 — High-Value Stable,Maintain Value,1557,22.11,30.80,21.53,4371.24,3.86,110,7.06,42982.02
2,2,Segment 2 — High-Value At-Risk,Protect High-Value,2260,32.09,57.24,87.39,4972.51,3.45,243,10.75,1696896.61


,state,customers,churned_customers,avg_cltv,total_revenue,avg_monthly_charge,avg_satisfaction,churn_rate_pct,revenue_at_risk,share_of_customers_pct
0,California,7043,1869,4400.3,21371131.69,64.76,3.24,26.54,3684459.82,100.0


,state,city,customers,churned_customers,avg_cltv,total_revenue,avg_monthly_charge,avg_satisfaction,churn_rate_pct,revenue_at_risk
0,California,Acampo,4,3,5254.5,18107.96,103.11,2.00,75.0,10471.36
1,California,Acton,4,0,4711.0,12156.36,69.53,4.25,0.0,0.00
2,California,Adelanto,5,1,4364.8,18235.49,56.56,3.60,20.0,4005.02
3,California,Adin,4,2,3990.5,5539.38,57.02,2.50,50.0,2457.91
4,California,Agoura Hills,5,2,4507.2,10641.88,48.99,2.80,40.0,9785.04


### Result & conclusion

I now have the three business-analysis grains separately: one row per segment, one row per state, and one row per state/city market. Keeping these grains separate prevents me from mixing customer-segment metrics with geographic metrics incorrectly.

In [6]:
segment_unique = segment_df["segment_id"].nunique()
segment_rows = len(segment_df)
state_unique = state_df["state"].nunique()
city_unique = city_df[["state", "city"]].drop_duplicates().shape[0]

segment_dup = int(segment_df["segment_id"].duplicated().sum())
state_dup = int(state_df["state"].duplicated().sum())
city_dup = int(city_df.duplicated(subset=["state", "city"]).sum())

assert segment_rows == segment_unique, "Duplicate segment IDs detected."
assert state_df["state"].notna().all(), "Null state values detected in state reporting table."
assert city_df[["state", "city"]].notna().all().all(), "Null state/city keys detected in city reporting table."
assert segment_dup == 0 and state_dup == 0 and city_dup == 0

print_result(
    f"Grain checks passed: {segment_unique} unique segments, {state_unique} states, {city_unique} state/city markets, and 0 duplicate keys."
)

Result: Grain checks passed: 3 unique segments, 1 states, 1106 state/city markets, and 0 duplicate keys.


### Result & conclusion

The reporting tables preserve their intended grains: one row per segment, one row per state, and one row per state/city combination. These checks reduce the risk of double counting exposure in the prioritization layer.

In [7]:
segment_exposure = (
    segment_df[["segment_id", "segment_name", "retention_profile", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk"]]
    .sort_values("revenue_at_risk", ascending=False)
    .reset_index(drop=True)
)

state_exposure = (
    state_df[["state", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk"]]
    .sort_values("revenue_at_risk", ascending=False)
    .reset_index(drop=True)
)

city_exposure = (
    city_df[["state", "city", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk"]]
    .sort_values("revenue_at_risk", ascending=False)
    .reset_index(drop=True)
)

print_result(
    f"Historical revenue-at-risk exposure totals ${segment_exposure['revenue_at_risk'].sum():,.2f} across segments; the same customer-level churned revenue is viewed again through state and city grains."
)
display(segment_exposure)

Result: Historical revenue-at-risk exposure totals $3,684,459.82 across segments; the same customer-level churned revenue is viewed again through state and city grains.


,segment_id,segment_name,retention_profile,customers,avg_cltv,churn_rate_pct,revenue_at_risk
0,0,Segment 0 — Emerging Risk,Retention Priority,3226,4013.45,46.99,1944581.19
1,2,Segment 2 — High-Value At-Risk,Protect High-Value,2260,4972.51,10.75,1696896.61
2,1,Segment 1 — High-Value Stable,Maintain Value,1557,4371.24,7.06,42982.02


### Result & conclusion

The same underlying customer exposure can legitimately be summarized by segment, state, or city. I will not add these tables together, because that would double-count the same customers across different dimensions.

In [8]:
score_file = CONFIG_DIR / "ease_of_intervention_scores.csv"

template = segment_df[["segment_id", "segment_name", "retention_profile"]].copy()
template["ease_of_intervention_score"] = np.nan
template["score_note"] = "Enter a subjective integer from 1 to 5; 5 means easier to intervene on operationally."

if score_file.exists():
    existing_scores = pd.read_csv(score_file)
    template = template.drop(columns=["ease_of_intervention_score", "score_note"])
    template = template.merge(existing_scores, on=["segment_id", "segment_name", "retention_profile"], how="left")
    template["score_note"] = template.get(
        "score_note",
        "Enter a subjective integer from 1 to 5; 5 means easier to intervene on operationally."
    )
else:
    template.to_csv(score_file, index=False)

# Keep the final template clean and stable.
if "ease_of_intervention_score" not in template.columns:
    template["ease_of_intervention_score"] = np.nan
if "score_note" not in template.columns:
    template["score_note"] = "Enter a subjective integer from 1 to 5; 5 means easier to intervene on operationally."
template.to_csv(score_file, index=False)

print_result(f"Prepared the editable segment intervention-score file at {score_file}.")
display(template)

Result: Prepared the editable segment intervention-score file at ..\config\ease_of_intervention_scores.csv.


,segment_id,segment_name,retention_profile,ease_of_intervention_score,score_note
0,0,Segment 0 — Emerging Risk,Retention Priority,NaN,Enter a subjective integer from 1 to 5; 5 mean...
1,1,Segment 1 — High-Value Stable,Maintain Value,NaN,Enter a subjective integer from 1 to 5; 5 mean...
2,2,Segment 2 — High-Value At-Risk,Protect High-Value,NaN,Enter a subjective integer from 1 to 5; 5 mean...


### Result & conclusion

I created or refreshed the **segment intervention-score template** without inventing the score. I need to enter a 1–5 business judgment for each segment before the weighted segment-priority analysis can be finalized.

**Important:** the score is an input assumption, not a statistic calculated from the dataset.

In [9]:
geography_score_file = CONFIG_DIR / "geography_ease_of_intervention_scores.csv"
geo_template = city_df[["state", "city"]].drop_duplicates().sort_values(["state", "city"]).copy()
geo_template["ease_of_intervention_score"] = np.nan
geo_template["score_note"] = "Optional: enter a subjective integer from 1 to 5 for geography-specific operational ease."

if geography_score_file.exists():
    existing_geo = pd.read_csv(geography_score_file)
    keep_cols = ["state", "city", "ease_of_intervention_score"]
    existing_geo = existing_geo[[c for c in keep_cols if c in existing_geo.columns]]
    geo_template = geo_template.drop(columns="ease_of_intervention_score").merge(
        existing_geo, on=["state", "city"], how="left"
    )

geo_template.to_csv(geography_score_file, index=False)
print_result(
    f"Prepared an optional geography score template with {len(geo_template):,} city rows. Geography scores are optional because the original roadmap only specifies scores per segment."
)
display(geo_template.head(20))

Result: Prepared an optional geography score template with 1,106 city rows. Geography scores are optional because the original roadmap only specifies scores per segment.


,state,city,ease_of_intervention_score,score_note
0,California,Acampo,NaN,Optional: enter a subjective integer from 1 to...
1,California,Acton,NaN,Optional: enter a subjective integer from 1 to...
2,California,Adelanto,NaN,Optional: enter a subjective integer from 1 to...
3,California,Adin,NaN,Optional: enter a subjective integer from 1 to...
4,California,Agoura Hills,NaN,Optional: enter a subjective integer from 1 to...
5,California,Aguanga,NaN,Optional: enter a subjective integer from 1 to...
6,California,Ahwahnee,NaN,Optional: enter a subjective integer from 1 to...
7,California,Alameda,NaN,Optional: enter a subjective integer from 1 to...
8,California,Alamo,NaN,Optional: enter a subjective integer from 1 to...
9,California,Albany,NaN,Optional: enter a subjective integer from 1 to...


### Result & conclusion

I also prepared an **optional** geography score template. I will use raw revenue-at-risk for geography prioritization when no geography-specific score is supplied, rather than incorrectly applying a segment score to an entire geography.

In [10]:
segment_df.to_csv(OUTPUT_DIR / "phase_07_segment_business_base.csv", index=False)
state_df.to_csv(OUTPUT_DIR / "phase_07_state_business_base.csv", index=False)
city_df.to_csv(OUTPUT_DIR / "phase_07_city_business_base.csv", index=False)

print_result("Saved the three Phase 7 business-analysis base tables to the outputs folder.")

Result: Saved the three Phase 7 business-analysis base tables to the outputs folder.


### Result & conclusion

The source datasets are now preserved as Phase 7 outputs. These files make the business-analysis layer reproducible even when someone reviews the project without opening the MySQL database.